# SpeechLM API in vLLM example

In [ ]:
from transformers import AutoModelForCausalLM
import torch

# load token embeddings and lm head to use outside of vllm
model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
hf_model = AutoModelForCausalLM.from_pretrained(model_name, dtype=torch.bfloat16)
embedding_layer = hf_model.get_input_embeddings().eval()
lm_head = hf_model.lm_head.eval()
del hf_model

In [ ]:
# example of llm streaming usage

from vllm import SamplingParams
from vllm.engine.arg_utils import AsyncEngineArgs
from vllm.v1.engine.async_llm import AsyncLLM

import time
import torch

engine_args = AsyncEngineArgs(
    model="TinyLlama/TinyLlama-1.1B-Chat-v1.0",
    max_model_len=256,
    gpu_memory_utilization=0.8,
    enable_prompt_embeds=True,
)
engine = AsyncLLM.from_engine_args(engine_args)
sampling_params = SamplingParams(max_tokens=20, temperature=1.0)

def top_p_sampling(logits, top_p=0.95, temperature=0.8, filter_value=-float('Inf')):
    logits = logits / temperature
    sorted_logits, sorted_indices = torch.sort(logits, descending=True, dim=-1)
    cumulative_probs = torch.cumsum(torch.nn.functional.softmax(sorted_logits, dim=-1), dim=-1)
    # Remove tokens with cumulative probability above the threshold
    sorted_indices_to_remove = cumulative_probs > top_p
    # Shift the indices to the right to keep first token above threshold
    sorted_indices_to_remove[..., 1:] = sorted_indices_to_remove[..., :-1].clone()
    sorted_indices_to_remove[..., 0] = 0
    indices_to_remove = sorted_indices[sorted_indices_to_remove]
    logits[0, indices_to_remove] = filter_value
    probs = torch.nn.functional.softmax(logits, dim=-1)  # 1 x VOCAB
    next_token = torch.multinomial(probs, num_samples=1).squeeze(dim=1)  # 1
    return next_token


tokens = []
async for output in engine.generate(request_id="1",prompt="Hi, I am", sampling_params=sampling_params):
    hidden_states = output.outputs[0].hidden_states # T x DIM  
    if hidden_states.shape[0] > 1:
        # use only last hidden state from the context phase
        hidden_states = hidden_states[-1:]  # 1 x DIM
    logits = lm_head(hidden_states)  # 1 x VOCAB
    next_token = top_p_sampling(logits)
    tokens.append(next_token.item())
    emb = embedding_layer(next_token).detach()  # 1 x DIM
    if not output.finished:
        await engine.append_request(request_id="1", input_embeds=emb)

print(f"Generated text: {engine.tokenizer.decode(tokens)}", flush=True)